# BOLT source evidence audit
Run from this notebook folder. Compare saved original source hashes, convert target coefficients, and inspect saved boundary diagnostics. For a full raw replay use the companion script with the original TDMS folders; this notebook does not fit parameters or validate a corrected model.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
folder=Path.cwd()
if not (folder/"summary.json").exists():
    folder=Path("research/cruise_m_build_20261011/runs/remaining_source_evidence_20261013")
summary=json.loads((folder/"summary.json").read_text())


In [2]:
conversion=np.array([4.4482216152605,4.4482216152605/1.609344,4.4482216152605/1.609344**2])
model=np.array([126.329494,0.557775,0.033456276])
for item in summary:
    gp=item["metadata"]
    target=np.array([float(gp["TestInfo_Dyno_Target_"+k]) for k in "ABC"])*conversion
    assert np.max(np.abs(target-model))<1e-6
    assert item["source_hash_matches_prior"]
    assert abs(item["test_mass_kg"]-1757.6704)<1e-3
    assert np.isclose(item["target_ABC_work_kWh"]-item["set_ABC_work_kWh"],item["target_minus_set_work_kWh"])
print("Saved source hashes, conversions and accounting checks passed")


Saved source hashes, conversions and accounting checks passed


In [3]:
pd.DataFrame([{k:item[k] for k in ["test","target_ABC_work_kWh","set_ABC_work_kWh","target_minus_set_work_kWh","dyno_front_force_speed_candidate_work_kWh"]} for item in summary])


## Interpretation
The target-versus-set difference includes the whole vehicle-on-dyno loss boundary. The complete zero-torque EDU electrical loss includes electrical losses and must not be subtracted as a neutral mechanical spin-loss correction. CAN current and torque remain unverified metrology boundaries.
